In [ ]:
"""
FDA NDC Directory Loader (Product + Package)
--------------------------------------------
Downloads the daily NDC text files from FDA,
builds standardized labeler_id and 11-digit NDC,
and writes both parquet and Excel outputs.
Excel NDC columns are forced to text to preserve leading zeros.
"""

import pandas as pd
import requests
import zipfile
import io
from pathlib import Path
from datetime import datetime
from config import out_folder

out_path = Path(out_folder)
out_path.mkdir(parents=True, exist_ok=True)

# Download text zip
url = "https://www.accessdata.fda.gov/cder/ndctext.zip"
print("Downloading NDC text zip...")
response = requests.get(url)
response.raise_for_status()

with zipfile.ZipFile(io.BytesIO(response.content)) as z:
    print("Files in zip:")
    for name in z.namelist():
        print(" ", name)

    # --- PACKAGE ---
    package_file = [n for n in z.namelist() if "package" in n.lower() and n.endswith(".txt")][0]
    with z.open(package_file) as f:
        df_package = pd.read_csv(f, sep="\t", dtype=str, low_memory=False, encoding="latin-1")

    # --- PRODUCT ---
    product_file = [n for n in z.namelist() if "product" in n.lower() and n.endswith(".txt")][0]
    with z.open(product_file) as f:
        df_product = pd.read_csv(f, sep="\t", dtype=str, low_memory=False, encoding="latin-1")

# ---------- Helpers ----------
def build_labeler(product_ndc):
    return product_ndc.astype(str).str.split("-").str[0].str.zfill(5)

def build_ndc11(ndc):
    parts = ndc.astype(str).str.split("-", expand=True)
    if parts.shape[1] == 3:
        labeler  = parts[0].str.zfill(5)
        product  = parts[1].str.zfill(4)
        package  = parts[2].str.zfill(2)
        return labeler + product + package
    return pd.Series([None] * len(ndc))

# ---------- Add columns ----------
load_ts = datetime.now()

df_package["labeler_id"] = build_labeler(df_package["PRODUCTNDC"])
df_package["ndc11"]      = build_ndc11(df_package["NDCPACKAGECODE"])
df_package["load_date"]  = load_ts

df_product["labeler_id"] = build_labeler(df_product["PRODUCTNDC"])
df_product["load_date"]  = load_ts

# ---------- Save ----------

# Comment Out xlsx files only use for viewability
df_package.to_excel(out_path / "ndc_package.xlsx", index=False)
df_package.to_parquet(out_path / "ndc_package.parquet", index=False)

# Comment Out xlsx files only use for viewability
df_product.to_excel(out_path / "ndc_product.xlsx", index=False)
df_product.to_parquet(out_path / "ndc_product.parquet", index=False)

print(f"Package: {df_package.shape[0]:,} rows → ndc_package.xlsx / .parquet")
print(f"Product: {df_product.shape[0]:,} rows → ndc_product.xlsx / .parquet")
print("Done.")

In [ ]:
"""
FDA Compounded Drugs NDC Directory Loader
-----------------------------------------
Downloads the daily compounders NDC file from FDA,
builds standardized labeler_id and 11-digit NDC,
and writes both parquet and Excel outputs.
Excel NDC columns are forced to text to preserve leading zeros.
"""

from pathlib import Path
from datetime import datetime
import zipfile
import requests
import pandas as pd
import openpyxl

from config import out_folder

out_path = Path(out_folder)
out_path.mkdir(parents=True, exist_ok=True)

# ---------- Download ----------
url = "https://www.accessdata.fda.gov/cder/compounders_ndc_directory.zip"
zip_path = out_path / "compounders_ndc_directory.zip"

print("Downloading compound NDC file...")
resp = requests.get(url, timeout=120)
resp.raise_for_status()
zip_path.write_bytes(resp.content)

with zipfile.ZipFile(zip_path, "r") as z:
    z.extract("compounders_ndc_directory.txt", out_path)

txt_path = out_path / "compounders_ndc_directory.txt"

# ---------- Load ----------
df = pd.read_csv(txt_path, sep="\t", dtype=str, low_memory=False)
print(f"Loaded {len(df):,} rows")

# ---------- Helpers ----------
def build_labeler(product_ndc):
    return product_ndc.astype(str).str.split("-").str[0].str.zfill(5)

def build_ndc11(ndc_package_code):
    parts = ndc_package_code.astype(str).str.split("-")
    labeler = parts.str[0].str.zfill(5)
    product = parts.str[1].str.zfill(4)
    package = parts.str[2].str.zfill(2)
    return labeler + product + package

df["labeler_id"] = build_labeler(df["PRODUCTNDC"])
df["ndc11"] = build_ndc11(df["NDCPACKAGECODE"])
df["load_date"] = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

# ---------- Save ----------
xlsx_path = out_path / "ndc_compound.xlsx"
parquet_path = out_path / "ndc_compound.parquet"

df.to_excel(xlsx_path, index=False)
df.to_parquet(parquet_path, index=False)

# Force NDC columns to text in Excel
def force_text_columns(filepath, columns):
    wb = openpyxl.load_workbook(filepath)
    ws = wb.active
    headers = [cell.value for cell in ws[1]]
    for col_name in columns:
        if col_name in headers:
            col_idx = headers.index(col_name) + 1
            for cell in ws.iter_cols(min_col=col_idx, max_col=col_idx, min_row=2):
                for c in cell:
                    c.number_format = "@"
                    if c.value is not None:
                        c.value = str(c.value)
    wb.save(filepath)

force_text_columns(xlsx_path, ["PRODUCTNDC", "NDCPACKAGECODE", "labeler_id", "ndc11"])

print(f"Compound: {df.shape[0]:,} rows → ndc_compound.xlsx / .parquet")
print("Done.")

In [ ]:
"""
FDA Unfinished Drugs NDC Directory Loader
-----------------------------------------
Downloads the daily unfinished NDC files from FDA,
builds standardized labeler_id (both files) and 11-digit NDC (package only),
and writes both parquet and Excel outputs.
Excel NDC columns are forced to text to preserve leading zeros.
"""

from pathlib import Path
from datetime import datetime
import zipfile
import requests
import pandas as pd
import openpyxl

from config import out_folder

out_path = Path(out_folder)
out_path.mkdir(parents=True, exist_ok=True)
print(f"Output folder: {out_path.resolve()}")

# ---------- Download ----------
url = "https://www.accessdata.fda.gov/cder/ndc_unfinished.zip"
zip_path = out_path / "ndc_unfinished.zip"

print("Downloading unfinished NDC files...")
resp = requests.get(url, timeout=120)
resp.raise_for_status()
zip_path.write_bytes(resp.content)

with zipfile.ZipFile(zip_path, "r") as z:
    z.extract("unfinished_product.txt", out_path)
    z.extract("unfinished_package.txt", out_path)
    product_modified = datetime(*z.getinfo("unfinished_product.txt").date_time).strftime("%Y-%m-%d %H:%M:%S")
    package_modified = datetime(*z.getinfo("unfinished_package.txt").date_time).strftime("%Y-%m-%d %H:%M:%S")

# ---------- Load (encoding-safe) ----------
def safe_read(path):
    for enc in ("utf-8", "latin-1", "cp1252"):
        try:
            return pd.read_csv(path, sep="\t", dtype=str, low_memory=False, encoding=enc)
        except Exception as e:
            print(f"  {enc} failed: {e}")
    raise RuntimeError(f"Could not read {path}")

print("Loading product...")
df_product = safe_read(out_path / "unfinished_product.txt")
print("Loading package...")
df_package = safe_read(out_path / "unfinished_package.txt")

print(f"Product: {len(df_product):,} rows")
print(f"Package: {len(df_package):,} rows")

# ---------- Helpers ----------
def build_labeler(product_ndc):
    return product_ndc.astype(str).str.split("-").str[0].str.zfill(5)

def build_ndc11(ndc_package_code):
    parts = ndc_package_code.astype(str).str.split("-")
    labeler = parts.str[0].str.zfill(5)
    product = parts.str[1].str.zfill(4)
    package = parts.str[2].str.zfill(2)
    return labeler + product + package

# ---------- Enrich ----------
df_product["labeler_id"] = build_labeler(df_product["PRODUCTNDC"])
df_product["load_date"] = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

df_package["labeler_id"] = build_labeler(df_package["PRODUCTNDC"])
df_package["ndc11"] = build_ndc11(df_package["NDCPACKAGECODE"])
df_package["load_date"] = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

# ---------- Save ----------
product_xlsx = out_path / "ndc_unfinished_product.xlsx"
package_xlsx = out_path / "ndc_unfinished_package.xlsx"

df_product.to_excel(product_xlsx, index=False)
df_product.to_parquet(out_path / "ndc_unfinished_product.parquet", index=False)

df_package.to_excel(package_xlsx, index=False)
df_package.to_parquet(out_path / "ndc_unfinished_package.parquet", index=False)

# Force NDC columns to text in Excel
def force_text_columns(filepath, columns):
    wb = openpyxl.load_workbook(filepath)
    ws = wb.active
    headers = [cell.value for cell in ws[1]]
    for col_name in columns:
        if col_name in headers:
            col_idx = headers.index(col_name) + 1
            for cell in ws.iter_cols(min_col=col_idx, max_col=col_idx, min_row=2):
                for c in cell:
                    c.number_format = "@"
                    if c.value is not None:
                        c.value = str(c.value)
    wb.save(filepath)

force_text_columns(product_xlsx, ["PRODUCTNDC", "labeler_id"])
force_text_columns(package_xlsx, ["PRODUCTNDC", "NDCPACKAGECODE", "labeler_id", "ndc11"])

print(f"\nFiles written to: {out_path.resolve()}")
print(f"  {product_xlsx.name}")
print(f"  ndc_unfinished_product.parquet")
print(f"  {package_xlsx.name}")
print(f"  ndc_unfinished_package.parquet")
print("Done.")

In [ ]:
"""
FDA Excluded Drugs NDC Directory Loader
---------------------------------------
Downloads the daily excluded NDC files from FDA,
builds standardized labeler_id (both files) and 11-digit NDC (package only),
and writes both parquet and Excel outputs.
Excel NDC columns are forced to text to preserve leading zeros.
"""

from pathlib import Path
from datetime import datetime
import zipfile
import requests
import pandas as pd
import openpyxl

from config import out_folder

out_path = Path(out_folder)
out_path.mkdir(parents=True, exist_ok=True)
print(f"Output folder: {out_path.resolve()}")

# ---------- Download ----------
url = "https://www.accessdata.fda.gov/cder/ndc_excluded.zip"
zip_path = out_path / "ndc_excluded.zip"

print("Downloading excluded NDC files...")
resp = requests.get(url, timeout=180)
resp.raise_for_status()
zip_path.write_bytes(resp.content)

with zipfile.ZipFile(zip_path, "r") as z:
    z.extract("Products_excluded.txt", out_path)
    z.extract("Packages_excluded.txt", out_path)
    product_modified = datetime(*z.getinfo("Products_excluded.txt").date_time).strftime("%Y-%m-%d %H:%M:%S")
    package_modified = datetime(*z.getinfo("Packages_excluded.txt").date_time).strftime("%Y-%m-%d %H:%M:%S")

# ---------- Load (encoding-safe) ----------
def safe_read(path):
    for enc in ("utf-8", "latin-1", "cp1252"):
        try:
            return pd.read_csv(path, sep="\t", dtype=str, low_memory=False, encoding=enc)
        except Exception as e:
            print(f"  {enc} failed: {e}")
    raise RuntimeError(f"Could not read {path}")

print("Loading product...")
df_product = safe_read(out_path / "Products_excluded.txt")
print("Loading package...")
df_package = safe_read(out_path / "Packages_excluded.txt")

print(f"Product: {len(df_product):,} rows")
print(f"Package: {len(df_package):,} rows")

# ---------- Helpers ----------
def build_labeler(product_ndc):
    return product_ndc.astype(str).str.split("-").str[0].str.zfill(5)

def build_ndc11(ndc_package_code):
    parts = ndc_package_code.astype(str).str.split("-")
    labeler = parts.str[0].str.zfill(5)
    product = parts.str[1].str.zfill(4)
    package = parts.str[2].str.zfill(2)
    return labeler + product + package

# ---------- Enrich ----------
df_product["labeler_id"] = build_labeler(df_product["PRODUCTNDC"])
df_product["load_date"] = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

df_package["labeler_id"] = build_labeler(df_package["PRODUCTNDC"])
df_package["ndc11"] = build_ndc11(df_package["NDCPACKAGECODE"])
df_package["load_date"] = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

# ---------- Save ----------
product_xlsx = out_path / "ndc_excluded_product.xlsx"
package_xlsx = out_path / "ndc_excluded_package.xlsx"

df_product.to_excel(product_xlsx, index=False)
df_product.to_parquet(out_path / "ndc_excluded_product.parquet", index=False)

df_package.to_excel(package_xlsx, index=False)
df_package.to_parquet(out_path / "ndc_excluded_package.parquet", index=False)

# Force NDC columns to text in Excel
def force_text_columns(filepath, columns):
    wb = openpyxl.load_workbook(filepath)
    ws = wb.active
    headers = [cell.value for cell in ws[1]]
    for col_name in columns:
        if col_name in headers:
            col_idx = headers.index(col_name) + 1
            for cell in ws.iter_cols(min_col=col_idx, max_col=col_idx, min_row=2):
                for c in cell:
                    c.number_format = "@"
                    if c.value is not None:
                        c.value = str(c.value)
    wb.save(filepath)

force_text_columns(product_xlsx, ["PRODUCTNDC", "labeler_id"])
force_text_columns(package_xlsx, ["PRODUCTNDC", "NDCPACKAGECODE", "labeler_id", "ndc11"])

print(f"\nFiles written to: {out_path.resolve()}")
print(f"  {product_xlsx.name}")
print(f"  ndc_excluded_product.parquet")
print(f"  {package_xlsx.name}")
print(f"  ndc_excluded_package.parquet")
print("Done.")

In [ ]:
### Quality Check: Core vs Non-Core Overlaps
"""
- Checks how many PRODUCTNDCs from unfinished, excluded, and compound
  also appear in the core product file
- Optional package-level checks on ndc11 (uncomment if desired)
- Uses parquet files already written to out_folder
"""

import duckdb
from pathlib import Path
from config import out_folder

con = duckdb.connect()
out_path = Path(out_folder)

# Load core tables
con.execute(f"""
CREATE OR REPLACE TABLE product AS 
SELECT * FROM read_parquet('{out_path / "ndc_product.parquet"}');

CREATE OR REPLACE TABLE package AS 
SELECT * FROM read_parquet('{out_path / "ndc_package.parquet"}');
""")

# Load non-core tables (add/remove as needed)
con.execute(f"""
CREATE OR REPLACE TABLE unfinished_product AS 
SELECT * FROM read_parquet('{out_path / "ndc_unfinished_product.parquet"}');

CREATE OR REPLACE TABLE excluded_product AS 
SELECT * FROM read_parquet('{out_path / "ndc_excluded_product.parquet"}');

CREATE OR REPLACE TABLE compound AS 
SELECT * FROM read_parquet('{out_path / "ndc_compound.parquet"}');

CREATE OR REPLACE TABLE unfinished_package AS 
SELECT * FROM read_parquet('{out_path / "ndc_unfinished_package.parquet"}');

CREATE OR REPLACE TABLE excluded_package AS 
SELECT * FROM read_parquet('{out_path / "ndc_excluded_package.parquet"}');
""")

print("Tables loaded.\n")

# --- Product-level overlaps (PRODUCTNDC) ---
print("=== PRODUCT OVERLAPS ===")

unfinished_in_core = con.execute("""
SELECT COUNT(DISTINCT u.PRODUCTNDC) 
FROM unfinished_product u
INNER JOIN product p ON u.PRODUCTNDC = p.PRODUCTNDC
""").fetchone()[0]
print(f"Unfinished products found in core product file: {unfinished_in_core:,}")

excluded_in_core = con.execute("""
SELECT COUNT(DISTINCT e.PRODUCTNDC) 
FROM excluded_product e
INNER JOIN product p ON e.PRODUCTNDC = p.PRODUCTNDC
""").fetchone()[0]
print(f"Excluded products found in core product file: {excluded_in_core:,}")

compound_in_core = con.execute("""
SELECT COUNT(DISTINCT c.PRODUCTNDC) 
FROM compound c
INNER JOIN product p ON c.PRODUCTNDC = p.PRODUCTNDC
""").fetchone()[0]
print(f"Compound products found in core product file: {compound_in_core:,}")

# --- Package-level overlaps (ndc11) ---
print("\n=== PACKAGE OVERLAPS ===")

unfinished_pkg_in_core = con.execute("""
SELECT COUNT(DISTINCT u.ndc11) 
FROM unfinished_package u
INNER JOIN package p ON u.ndc11 = p.ndc11
""").fetchone()[0]
print(f"Unfinished packages found in core package file: {unfinished_pkg_in_core:,}")

excluded_pkg_in_core = con.execute("""
SELECT COUNT(DISTINCT e.ndc11) 
FROM excluded_package e
INNER JOIN package p ON e.ndc11 = p.ndc11
""").fetchone()[0]
print(f"Excluded packages found in core package file: {excluded_pkg_in_core:,}")

print("\nDone.")

In [ ]:
# Load excluded package (adjust filename if needed)
con.execute("DROP TABLE IF EXISTS excluded_package")
con.execute(f"""
CREATE TABLE excluded_package AS 
SELECT * FROM read_parquet('{out_path / "ndc_excluded_package.parquet"}');
""")

query = """
SELECT 
    p.ndc11,
    p.NDCPACKAGECODE,
    COALESCE(p.PACKAGEDESCRIPTION, '')          AS package_description,
    p.STARTMARKETINGDATE                        AS package_start_date,
    COALESCE(p.ENDMARKETINGDATE, '')            AS package_end_date,
    p.labeler_id,
    p.load_date,
    
    pr.PRODUCTNDC,
    COALESCE(pr.PROPRIETARYNAME, '')            AS proprietary_name,
    COALESCE(pr.NONPROPRIETARYNAME, '')         AS nonproprietary_name,
    COALESCE(pr.DOSAGEFORMNAME, '')             AS dosage_form,
    COALESCE(pr.ROUTENAME, '')                  AS route,
    COALESCE(pr.SUBSTANCENAME, '')              AS substance,
    COALESCE(pr.ACTIVE_NUMERATOR_STRENGTH, '')  AS strength,
    COALESCE(pr.ACTIVE_INGRED_UNIT, '')         AS strength_unit,
    COALESCE(pr.PHARM_CLASSES, '')              AS pharm_classes,
    COALESCE(pr.DEASCHEDULE, '')                AS dea_schedule,
    COALESCE(pr.LABELERNAME, '')                AS labeler_name,
    COALESCE(pr.MARKETINGCATEGORYNAME, '')      AS marketing_category,
    pr.STARTMARKETINGDATE                       AS product_start_date,
    COALESCE(pr.ENDMARKETINGDATE, '')           AS product_end_date,


FROM package p
LEFT JOIN product pr 
    ON p.PRODUCTID = pr.PRODUCTID
"""

df = con.execute(query).df()

def widen_display(max_colwidth=80, width=1000):
    pd.set_option('display.max_columns', None)
    pd.set_option('display.width', width)
    pd.set_option('display.max_colwidth', max_colwidth)

# Usage
widen_display()
print(df.head(10).to_string())
# print(f"Joined rows: {len(df):,}")
# print(df.head(10))
# print(f"Excluded packages found in core: {df['is_excluded'].sum():,}")
# Export to Excel in the output folder
# output_file = out_path / "ndc_package_product_with_excluded_flag.xlsx"
# df.to_excel(output_file, index=False)

# print(f"Exported to: {output_file}")

In [ ]:
# =============================================================================
# RxNorm NDC Crosswalk
# =============================================================================
# Dynamically finds and downloads the latest RxNorm Full Prescribe zip
# from the NLM RxNorm Files page if it doesn't already exist in out_folder.
#
# Extracts the NDC-to-RxCUI crosswalk from the RXNSAT.RRF file
# (ATN = 'NDC' and SAB = 'RXNORM').
#
# Writes a clean one-to-many mapping of NDC-11 → RxCUI to parquet
# so it can be queried later with DuckDB.
# =============================================================================

import pandas as pd
import zipfile
import requests
from pathlib import Path
from datetime import datetime
from bs4 import BeautifulSoup
from config import out_folder

out_path = Path(out_folder)
out_path.mkdir(parents=True, exist_ok=True)

zip_candidates = list(out_path.glob("RxNorm_full_prescribe_*.zip"))

if zip_candidates:
    zip_path = zip_candidates[0]
    print(f"Using existing: {zip_path.name}")
else:
    page_url = "https://www.nlm.nih.gov/research/umls/rxnorm/docs/rxnormfiles.html"
    resp = requests.get(page_url)
    resp.raise_for_status()
    soup = BeautifulSoup(resp.text, "html.parser")

    link = soup.find("a", href=lambda href: href and "RxNorm_full_prescribe_" in href and href.endswith(".zip"))
    if not link:
        raise ValueError("Could not find latest RxNorm prescribe zip link.")

    url = link["href"]
    zip_name = url.split("/")[-1]
    zip_path = out_path / zip_name

    print(f"Downloading: {url}")
    r = requests.get(url, stream=True)
    r.raise_for_status()
    with open(zip_path, "wb") as f:
        for chunk in r.iter_content(chunk_size=1024*1024):
            f.write(chunk)
    print("Download complete.")

load_date = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

with zipfile.ZipFile(zip_path) as z:
    sat = pd.read_csv(
        z.open("rrf/RXNSAT.RRF"),
        sep="|",
        header=None,
        dtype=str,
        usecols=[0, 8, 9, 10],
        names=["rxcui", "atn", "sab", "ndc11"]
    )

ndc_xwalk = (
    sat[(sat["atn"] == "NDC") & (sat["sab"] == "RXNORM")]
    [["ndc11", "rxcui"]]
    .drop_duplicates()
    .reset_index(drop=True)
)

ndc_xwalk["load_date"] = load_date
ndc_xwalk.to_parquet(out_path / "ndc_xwalk.parquet", index=False)

print(ndc_xwalk.shape)
print(ndc_xwalk.head())
print("Wrote ndc_xwalk.parquet")

In [ ]:
# =============================================================================
# RxNorm Concepts
# =============================================================================
# Reads the previously downloaded RxNorm Full Prescribe zip
# (RxNorm_full_prescribe_*.zip) from the configured out_folder.
#
# Extracts preferred concept names from the RXNCONSO.RRF file
# for the most useful term types (SCD, SBD, GPCK, BPCK, IN, PIN, MIN, BN).
#
# Adds a human-readable term_type lookup and writes the result to parquet
# so it can be queried later with DuckDB.
# =============================================================================

import pandas as pd
import zipfile
from pathlib import Path
from datetime import datetime
from config import out_folder

out_path = Path(out_folder)

zip_candidates = list(out_path.glob("RxNorm_full_prescribe_*.zip"))
if not zip_candidates:
    raise FileNotFoundError(f"No RxNorm_full_prescribe_*.zip found in {out_folder}")

zip_path = zip_candidates[0]
print(f"Using: {zip_path.name}")

load_date = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

with zipfile.ZipFile(zip_path) as z:
    conso = pd.read_csv(
        z.open("rrf/RXNCONSO.RRF"),
        sep="|",
        header=None,
        dtype=str,
        usecols=[0, 11, 12, 13, 14],
        names=["rxcui", "suppress", "tty", "code", "str"]
    )

concepts = (
    conso[conso["tty"].isin(["SCD", "SBD", "GPCK", "BPCK", "IN", "PIN", "MIN", "BN"])]
    .drop_duplicates(subset="rxcui")
    [["rxcui", "str", "tty", "suppress"]]
    .rename(columns={"str": "name"})
    .reset_index(drop=True)
)

concepts["load_date"] = load_date

# TTY lookup
tty_lookup = pd.DataFrame({
    "tty": [
        "SCD", "SBD", "GPCK", "BPCK",
        "IN", "PIN", "BN", "DF",
        "SCDF", "SBDF", "SCDG", "SBDG",
        "MIN", "TMSY", "SY", "PSN"
    ],
    "term_type": [
        "Semantic Clinical Drug",
        "Semantic Branded Drug",
        "Generic Pack",
        "Branded Pack",
        "Ingredient",
        "Precise Ingredient",
        "Brand Name",
        "Dose Form",
        "Semantic Clinical Drug Form",
        "Semantic Branded Drug Form",
        "Semantic Clinical Drug Group",
        "Semantic Branded Drug Group",
        "Multiple Ingredients",
        "Tall Man Synonym",
        "Synonym",
        "Prescribable Name"
    ]
})

concepts = concepts.merge(tty_lookup, on="tty", how="left")

concepts.to_parquet(out_path / "concepts.parquet", index=False)

print(concepts.shape)
print(concepts.head())
print("Wrote concepts.parquet")

In [ ]:
# =============================================================================
# Load parquet tables and run the NDC + RxNorm join
# =============================================================================
# Expects the following parquet files to already exist in out_folder:
#   - ndc_package.parquet
#   - ndc_product.parquet
#   - ndc_xwalk.parquet
#   - concepts.parquet
# =============================================================================

import pandas as pd
import duckdb
from pathlib import Path
from config import out_folder

out_path = Path(out_folder)
con = duckdb.connect()

con.execute(f"""
CREATE OR REPLACE TABLE package AS 
SELECT * FROM read_parquet('{out_path / "ndc_package.parquet"}');

CREATE OR REPLACE TABLE product AS 
SELECT * FROM read_parquet('{out_path / "ndc_product.parquet"}');

CREATE OR REPLACE TABLE ndc_xwalk AS 
SELECT * FROM read_parquet('{out_path / "ndc_xwalk.parquet"}');

CREATE OR REPLACE TABLE concepts AS 
SELECT * FROM read_parquet('{out_path / "concepts.parquet"}');
""")

query = """
SELECT 
    p.ndc11,
    p.NDCPACKAGECODE,
    COALESCE(p.PACKAGEDESCRIPTION, '')          AS package_description,
    p.STARTMARKETINGDATE                        AS package_start_date,
    COALESCE(p.ENDMARKETINGDATE, '')            AS package_end_date,
    p.labeler_id,
    p.load_date                                 AS package_load_date,

    pr.PRODUCTNDC,
    COALESCE(pr.PROPRIETARYNAME, '')            AS proprietary_name,
    COALESCE(pr.NONPROPRIETARYNAME, '')         AS nonproprietary_name,
    COALESCE(pr.DOSAGEFORMNAME, '')             AS dosage_form,
    COALESCE(pr.ROUTENAME, '')                  AS route,
    COALESCE(pr.SUBSTANCENAME, '')              AS substance,
    COALESCE(pr.ACTIVE_NUMERATOR_STRENGTH, '')  AS strength,
    COALESCE(pr.ACTIVE_INGRED_UNIT, '')         AS strength_unit,
    COALESCE(pr.PHARM_CLASSES, '')              AS pharm_classes,
    COALESCE(pr.DEASCHEDULE, '')                AS dea_schedule,
    COALESCE(pr.LABELERNAME, '')                AS labeler_name,
    COALESCE(pr.MARKETINGCATEGORYNAME, '')      AS marketing_category,
    pr.STARTMARKETINGDATE                       AS product_start_date,
    COALESCE(pr.ENDMARKETINGDATE, '')           AS product_end_date,

    x.rxcui,
    x.load_date                                 AS xref_load_date,

    c.name                                      AS rxnorm_name,
    c.tty,
    c.term_type,
    c.suppress,
    c.load_date                                 AS concept_load_date

FROM package p
INNER JOIN product pr 
    ON p.PRODUCTID = pr.PRODUCTID
INNER JOIN ndc_xwalk x 
    ON p.ndc11 = x.ndc11
INNER JOIN concepts c 
    ON x.rxcui = c.rxcui
"""

df = con.execute(query).df()

def widen_display(max_colwidth=80, width=1200):
    pd.set_option('display.max_columns', None)
    pd.set_option('display.width', width)
    pd.set_option('display.max_colwidth', max_colwidth)

widen_display()
print(df.head(10).to_string())
print(f"\nTotal rows: {len(df):,}")

excel_path = out_path / "ndc_rxnorm_join.xlsx"

with pd.ExcelWriter(excel_path, engine="openpyxl") as writer:
    df.to_excel(writer, index=False, sheet_name="ndc_rxnorm")
    worksheet = writer.sheets["ndc_rxnorm"]
    for col in worksheet.columns:
        worksheet.column_dimensions[col[0].column_letter].width = 50

print(f"Wrote {len(df):,} rows to {excel_path}")